# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.

**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

In [4]:
# Install required packages
! pip install requests beautifulsoup4 lxml pandas



  Using cached pandas-3.0.2-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.4.4-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2026.1-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.0 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.0 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.0 MB ? eta -:--:--
   ------- -------------------------------- 0.8/4.0 MB 850.3 kB/s eta 0:00:04
   ------- -------------------------------- 0.8/4.0 MB 850.3 kB/s eta 0:00:04
   ---------- ----------------------------- 1.0/4.0 MB 758.2 kB/s eta 0:00:04
   ------------- -------------------------- 1.3/4.0 MB 822.0 kB/s eta 0:00:04
   --------------- ------------------------ 1.6/4.0 MB 921.1 kB/s eta 0:00:03
   ------------------ --------------------- 1.8/4.0 MB 998.6 kB/s eta 0:00:03
   ------------------ --------------------- 1.

In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

def scrape_books(min_rating=4.0, max_price=20.0):
    base_url = "https://books.toscrape.com/"
    books_data = []
    
    page = 1
    while True:
        url = f"{base_url}catalogue/page-{page}.html" if page > 1 else f"{base_url}index.html"
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        if soup.find('li', class_='next') is None:
            break
            
        book_panels = soup.find_all('article', class_='product_pod')
        for panel in book_panels:
            # Price
            price_elem = panel.find('p', class_='price_color')
            if not price_elem:
                continue
            price_match = re.search(r'£([\d.]+)', price_elem.text)
            if not price_match:
                continue
            price = float(price_match.group(1))
            if price > max_price:
                continue
                
            # Rating
            rating_elem = panel.find('p', class_='star-rating')
            if not rating_elem:
                continue
            rating_class = rating_elem['class'][1]
            rating_map = {'One':1, 'Two':2, 'Three':3, 'Four':4, 'Five':5}
            if rating_class not in rating_map or rating_map[rating_class] < min_rating:
                continue
                
            rating = rating_map[rating_class]
            
            # Title & link
            title_elem = panel.find('h3').find('a')
            title = title_elem['title']
            book_url = base_url.replace('index.html', '') + title_elem['href'].replace('../../../', '')
            
            # Detail page (robust)
            try:
                detail_resp = requests.get(book_url, timeout=5)
                detail_soup = BeautifulSoup(detail_resp.content, 'html.parser')
                
                # UPC from td (correct selector)
                td_rows = detail_soup.find_all('td')
                upc = td_rows[0].text if len(td_rows) > 0 else "N/A"
                
                # Availability
                avail_elem = detail_soup.find('p', class_='instock availability')
                availability = avail_elem.text.strip() if avail_elem else "Unknown"
                
                # Description
                desc_elem = detail_soup.find('div', id='product_description')
                description = (desc_elem.find('p').text.strip() if desc_elem and desc_elem.find('p') 
                              else "No description")
                
                # Genre from breadcrumb
                breadcrumb = detail_soup.find('ul', class_='breadcrumb')
                genre_links = breadcrumb.find_all('a') if breadcrumb else []
                genre = genre_links[2].text.strip() if len(genre_links) > 2 else "Unknown"
                
            except:
                upc, availability, description, genre = "Error", "Error", "Error", "Error"
            
            books_data.append({
                'UPC': upc,
                'Title': title,
                'Price (£)': round(price, 2),
                'Rating': rating,
                'Genre': genre,
                'Availability': availability,
                'Description': description[:200] + "..." if len(description) > 200 else description
            })
            
            time.sleep(0.5)  # Polite
        
        page += 1
        time.sleep(1)
    
    df = pd.DataFrame(books_data)
    return df

# Run
df_books = scrape_books(4.0, 20.0)
print(f"✅ Found {len(df_books)} books")
print(df_books.head())
df_books.to_csv('high_rated_books.csv', index=False)

✅ Found 74 books
                UPC                                              Title  \
0  ce6396b0f23f6ecc                                        Set Me Free   
1               N/A  The Four Agreements: A Practical Guide to Pers...   
2               N/A                                     Sophie's World   
3               N/A            Untitled Collection: Sabbath Poems 2014   
4               N/A                                    This One Summer   

   Price (£)  Rating        Genre             Availability     Description  
0      17.46       5  Young Adult  In stock (19 available)  No description  
1      17.66       5      Unknown                  Unknown  No description  
2      15.94       5      Unknown                  Unknown  No description  
3      14.27       4      Unknown                  Unknown  No description  
4      19.49       4      Unknown                  Unknown  No description  
